In [ ]:
import os, time; os.environ["SESSION_T0"] = str(time.time())
import json
from pathlib import Path

RUN = {
    "config": "pipeline/multi_lora_sft/configs/31b_full.yaml",
    "train_model_hint": "31b-it-qat-q4_0-unquantized",
    "served_model_hint": "31b-it-qat-w4a16-ct",
    "train_files": [],          # override via --set train_files=[...] or auto-discovered *.jsonl
    "eval_task_ids": [],        # 3 fixed held-out task IDs from tasks.jsonl
    "set": [],                  # extra dotted overrides passed to pipeline.multi_lora_sft.run
}
RUN_OVERRIDES = {}  # @@RUN_OVERRIDES@@  (rewritten by scripts/kaggle_push.py)
RUN.update(RUN_OVERRIDES)

INPUT = Path("/kaggle/input")
SKIP_DIRS = {"snapshots", "graphs", "embeddings", ".git", "__pycache__"}

def find_dirs(pred, root=INPUT, max_depth=7):
    hits = []
    if not root.exists():
        return hits
    for dirpath, dirnames, filenames in os.walk(root):
        d = Path(dirpath)
        if pred(d, set(filenames)):
            hits.append(d)
            dirnames[:] = []
            continue
        depth = len(d.relative_to(root).parts)
        dirnames[:] = [] if depth >= max_depth else sorted(n for n in dirnames if n not in SKIP_DIRS)
    return hits

COMP = (find_dirs(lambda d, f: {"tasks.jsonl", "HARNESS_README.md"} <= f) or [None])[0]
REPO_SRC = (find_dirs(lambda d, f: (d / "pipeline" / "multi_lora_sft" / "run.py").is_file()) or [Path.cwd()])[0]
MODELS = find_dirs(lambda d, f: "config.json" in f and any(n.endswith(".safetensors") for n in f))
WHEELHOUSES = [w for w in find_dirs(lambda d, f: any(n.endswith(".whl") for n in f)) if not (COMP and w.is_relative_to(COMP))]

TRAIN_MODEL = next((m for m in MODELS if RUN["train_model_hint"].lower() in str(m).lower()), None)
SERVED_MODEL = next((m for m in MODELS if RUN["served_model_hint"].lower() in str(m).lower()), None)

print(f"SESSION_T0 = {os.environ['SESSION_T0']}")
print(json.dumps({"run": RUN, "comp": str(COMP), "repo": str(REPO_SRC), "train_model": str(TRAIN_MODEL), "served_model": str(SERVED_MODEL), "wheelhouses": [str(w) for w in WHEELHOUSES]}, indent=2))


In [ ]:
import importlib.metadata
import shutil
import subprocess
import sys

OFFICIAL_WH = [w for w in WHEELHOUSES if "developer-agent" in str(w).lower() or "vllm" in " ".join(p.name for p in w.glob("*.whl")).lower()]
PRIVATE_WH = [w for w in WHEELHOUSES if w not in OFFICIAL_WH]
STAGING = Path("/tmp/wheelhouse_staging")
STAGING.mkdir(parents=True, exist_ok=True)

def stage_and_install(wh: Path, no_deps: bool = False):
    dst_dir = STAGING / wh.name
    dst_dir.mkdir(parents=True, exist_ok=True)
    staged = []
    for w in sorted(wh.glob("*.whl")):
        # Restore +cu128 local version suffix if stripped by Kaggle dataset upload
        name = w.name.replace("cu128", "+cu128", 1) if (w.name.startswith("torch-") and "cu128" in w.name and "+cu128" not in w.name) else w.name
        target = dst_dir / name
        if not target.exists():
            shutil.copy2(w, target)
        staged.append(str(target))
    if not staged:
        return
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", str(dst_dir)]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(staged)
    subprocess.run(cmd, check=True)

for wh in OFFICIAL_WH:
    print(f"Installing official wheelhouse: {wh}")
    stage_and_install(wh, no_deps=False)
for wh in PRIVATE_WH:
    print(f"Installing training wheelhouse (--no-deps): {wh}")
    stage_and_install(wh, no_deps=True)

for pkg in ("torch", "transformers", "peft", "trl", "accelerate", "datasets", "bitsandbytes", "vllm", "swegemma", "adk-submission"):
    try:
        print(f"  {pkg:<18} {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {pkg:<18} (not installed)")


In [ ]:
REPO_WORK = Path("/tmp/GemmaFineTuning")
if REPO_SRC.resolve() != REPO_WORK.resolve():
    shutil.rmtree(REPO_WORK, ignore_errors=True)
    shutil.copytree(REPO_SRC, REPO_WORK, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".git", "runs"))
os.chdir(REPO_WORK)

cmd = [sys.executable, "-m", "pipeline.multi_lora_sft.run", "--config", str(REPO_WORK / RUN["config"])]
if RUN.get("git_sha"):
    cmd += ["--set", f"run.git_sha={RUN['git_sha']}"]
if TRAIN_MODEL is not None:
    cmd += ["--set", f"model.train_base_path={TRAIN_MODEL}"]
if SERVED_MODEL is not None:
    cmd += ["--set", f"model.served_path={SERVED_MODEL}"]
if COMP is not None:
    cmd += ["--set", f"data.tasks_file={COMP / 'tasks.jsonl'}"]
    for sub_key, folder in (("snapshots_dir", "snapshots"), ("graphs_dir", "graphs"), ("embeddings_dir", "embeddings"), ("wheels_dir", "wheels")):
        if (COMP / folder).is_dir():
            cmd += ["--set", f"eval.{sub_key}={COMP / folder}"]
if RUN.get("train_files"):
    cmd += ["--set", f"data.train_files={json.dumps(RUN['train_files'])}"]
if RUN.get("eval_task_ids"):
    cmd += ["--set", f"eval.task_ids={json.dumps(RUN['eval_task_ids'])}"]
for expr in RUN.get("set", []):
    cmd += ["--set", str(expr)]

print("$ " + " ".join(cmd))
ORCH_PROC = subprocess.run(cmd)
print(f"Orchestrator exited with code {ORCH_PROC.returncode}")


In [ ]:
RUNS_ROOT = Path("/kaggle/working/runs")
reports = sorted(RUNS_ROOT.glob("*/REPORT.md"), key=lambda p: p.stat().st_mtime) if RUNS_ROOT.exists() else []
if reports:
    latest_report = reports[-1]
    print(f"=== {latest_report} ===\n")
    print(latest_report.read_text(encoding="utf-8"))
else:
    print("No REPORT.md found under /kaggle/working/runs")

work_zip = Path("/kaggle/working/submission.zip")
print(f"\nsubmission.zip present: {work_zip.is_file()}" + (f" ({work_zip.stat().st_size / (1024 * 1024):.2f} MiB)" if work_zip.is_file() else ""))
if ORCH_PROC.returncode != 0:
    raise RuntimeError(f"Multi-LoRA SFT pipeline exited with code {ORCH_PROC.returncode}")
